In [1]:
import os
import sqlite3
from pathlib import Path
from datetime import datetime


In [2]:
# Get the path of the notebook
current_notebook = Path(__file__).resolve() if '__file__' in globals() else Path().resolve()

# Traverse up to DE_Tools
ROOT_DIR = current_notebook.parents[3]  # Go up 4 levels to get to DE_Tools

# Define where to store the SQLite database
DB_PATH = current_notebook/ "folder_structure.db"

print(f"Scanning root: {ROOT_DIR}")
print(f"Saving DB to:  {DB_PATH}")


Scanning root: C:\Users\RhysL\Desktop\DE_Tools
Saving DB to:  C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools\Folder-DB\folder_structure.db


In [3]:

# --- Connect to SQLite3 database ---
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [4]:

# --- Create schema ---
cursor.execute('''
CREATE TABLE IF NOT EXISTS folders (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    path TEXT NOT NULL,
    parent_id INTEGER,
    FOREIGN KEY(parent_id) REFERENCES folders(id)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS files (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    path TEXT NOT NULL,
    folder_id INTEGER,
    extension TEXT,
    size INTEGER,
    modified_time TEXT,
    created_time TEXT,
    FOREIGN KEY(folder_id) REFERENCES folders(id)
)
''')

conn.commit()


In [5]:
def insert_folder_structure(base_path: Path):
    folder_id_map = {}

    for dirpath, dirnames, filenames in os.walk(base_path):
        dir_path = Path(dirpath)
        parent_path = dir_path.parent
        parent_id = folder_id_map.get(parent_path.as_posix())

        # Skip hidden directories in-place (modifies the list os.walk will recurse into)
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]

        # Skip the current folder itself if hidden
        if dir_path.name.startswith("."):
            continue

        # Insert folder
        cursor.execute(
            "INSERT INTO folders (name, path, parent_id) VALUES (?, ?, ?)",
            (dir_path.name, dir_path.as_posix(), parent_id)
        )
        folder_id = cursor.lastrowid
        folder_id_map[dir_path.as_posix()] = folder_id

        # Filter hidden files
        visible_files = [f for f in filenames if not f.startswith(".")]

        for fname in visible_files:
            fpath = dir_path / fname
            try:
                stat = fpath.stat()
            except FileNotFoundError:
                continue  # skip broken symlinks or race conditions

            cursor.execute(
                '''INSERT INTO files 
                   (name, path, folder_id, extension, size, modified_time, created_time) 
                   VALUES (?, ?, ?, ?, ?, ?, ?)''',
                (
                    fpath.name,
                    fpath.as_posix(),
                    folder_id,
                    fpath.suffix,
                    stat.st_size,
                    datetime.fromtimestamp(stat.st_mtime).isoformat(),
                    datetime.fromtimestamp(stat.st_ctime).isoformat()
                )
            )

    conn.commit()


In [6]:
# --- Run the process ---
if __name__ == "__main__":
    if not ROOT_DIR.exists():
        print(f"Root folder does not exist: {ROOT_DIR}")
    else:
        print(f"Scanning: {ROOT_DIR}")
        insert_folder_structure(ROOT_DIR)
        print("Done.")
        print(f"Database saved to: {DB_PATH.resolve()}")


Scanning: C:\Users\RhysL\Desktop\DE_Tools
Done.
Database saved to: C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools\Folder-DB\folder_structure.db


In [7]:
cursor.close()

## SQL Queries

In [8]:
import sqlite3
import pandas as pd
from pathlib import Path

In [12]:

# Path to your SQLite database
DB_PATH = Path("folder_structure.db")
conn = sqlite3.connect(DB_PATH)

def run_query(query, params=None, description=None):
    if description:
        print(f"--- {description} --- \n")
    df = pd.read_sql(query, conn, params=params)
    print(df, end="\n\n")


In [13]:
# --- Basic queries ---

run_query("""
SELECT 
    (SELECT COUNT(*) FROM folders) AS total_folders,
    (SELECT COUNT(*) FROM files) AS total_files;
""", description="1. Count total folders and files")


--- 1. Count total folders and files --- 

   total_folders  total_files
0           2928        23994



In [14]:
run_query("""
SELECT id, name, path 
FROM folders 
WHERE parent_id IS NULL;
""", description="2. List top-level folders")

--- 2. List top-level folders --- 

   id      name                             path
0   1  DE_Tools  C:/Users/RhysL/Desktop/DE_Tools



In [15]:
run_query("""
SELECT f.name, f.path, COUNT(*) AS file_count
FROM folders f
JOIN files fi ON f.id = fi.folder_id
GROUP BY f.id
ORDER BY file_count DESC
LIMIT 10;
""", description="3. Folders with most files")


--- 3. Folders with most files --- 

             name                                               path  \
0       documents  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
1     __pycache__  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
2          lexers  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
3  JMulTi_results  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
4         America  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
5         America  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
6      matplotlib  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
7           2and3  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
8  pyasn1_modules  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   
9               2  C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...   

   file_count  
0         535  
1         258  
2         258  
3         175  
4         143  
5         142  
6         141  
7         133  
8         132  
9         

In [16]:
run_query("""
SELECT extension, COUNT(*) AS count
FROM files
GROUP BY extension
ORDER BY count DESC;
""", description="4. Count files by extension")


--- 4. Count files by extension --- 

    extension  count
0         .py  10636
1        .pyc   5276
2        .pyi   2272
3               1782
4       .json    566
..        ...    ...
121     .bash      1
122      .ani      1
123      .PSF      1
124      .MIT      1
125  .APACHE2      1

[126 rows x 2 columns]



In [24]:
run_query("""
SELECT folders.name AS folder_name, files.name, files.path, files.size
FROM files
JOIN folders ON files.folder_id = folders.id
ORDER BY files.size DESC
LIMIT 10;
""", description="5. Largest files with folder name")


--- 5. Largest files with folder name --- 

       folder_name                                               name  \
0  MultiProcessing               photo-1493976040374-85c8e12f0c0e.jpg   
1        Threading               photo-1493976040374-85c8e12f0c0e.jpg   
2       numpy.libs  libscipy_openblas64_-43e11ff0749b8cbe0a615c9cf...   
3       scipy.libs  libscipy_openblas-f07f5a5d207a3a47104dca54d6d0...   
4  MultiProcessing               photo-1541698444083-023c97d3f4b6.jpg   
5        Threading               photo-1541698444083-023c97d3f4b6.jpg   
6  MultiProcessing               photo-1532009324734-20a7a5813719.jpg   
7        Threading               photo-1532009324734-20a7a5813719.jpg   
8  MultiProcessing               photo-1522364723953-452d3431c267.jpg   
9        Threading               photo-1522364723953-452d3431c267.jpg   

                                                path      size  
0  C:/Users/RhysL/Desktop/DE_Tools/Reference/Snip...  21040337  
1  C:/Users/RhysL/Desk

In [ ]:

run_query("""
SELECT extension, AVG(size) AS avg_size, COUNT(*) AS count
FROM files
GROUP BY extension
ORDER BY avg_size DESC;
""", description="6. Average file size by extension")


In [ ]:

run_query("""
SELECT name, path, modified_time
FROM files
ORDER BY modified_time DESC
LIMIT 10;
""", description="7. Files modified recently")


In [ ]:

# --- Queries with variables for flexible exploration ---

# Variable: Folder path prefix to filter folders (and files within)
folder_path_prefix = '/Users/yourname/Documents'  # Adjust as needed

query_folder_files = """
SELECT f.name AS folder_name, fi.name AS file_name, fi.path AS file_path
FROM folders f
JOIN files fi ON f.id = fi.folder_id
WHERE f.path LIKE ?
ORDER BY f.path, fi.name
LIMIT 20;
"""
run_query(query_folder_files, params=(folder_path_prefix + '%',), description=f"8. Files in folders starting with '{folder_path_prefix}'")

# Variable: File extension to filter
file_extension = '.md'

query_files_by_extension = """
SELECT name, path, size, modified_time
FROM files
WHERE extension = ?
ORDER BY modified_time DESC
LIMIT 20;
"""
run_query(query_files_by_extension, params=(file_extension,), description=f"9. Most recent files with extension '{file_extension}'")

# Variable: File name to search exact matches
file_name_search = 'README.md'

query_files_by_name = """
SELECT name, path, size, modified_time
FROM files
WHERE name = ?
ORDER BY modified_time DESC
LIMIT 10;
"""
run_query(query_files_by_name, params=(file_name_search,), description=f"10. Files named '{file_name_search}'")

# Variable: Minimum file size to filter large files (bytes)
min_size = 1_000_000  # 1 MB

query_large_files = """
SELECT name, path, size
FROM files
WHERE size > ?
ORDER BY size DESC
LIMIT 20;
"""
run_query(query_large_files, params=(min_size,), description=f"11. Files larger than {min_size} bytes")

# Variable: Count files grouped by year of modification
query_files_by_year = """
SELECT SUBSTR(modified_time, 1, 4) AS year, COUNT(*) AS count
FROM files
GROUP BY year
ORDER BY year DESC;
"""
run_query(query_files_by_year, description="12. Count of files by year")

conn.close()
